# TTS Exp1 — Step 0: Pre-generate LLM Responses

Chạy notebook này **1 lần duy nhất** để generate 100 LLM responses.  
Output: `llm_responses.json` → upload lên Drive → các notebook A/B/C load lại.

**Lý do tách riêng:** vLLM conflict với lmdeploy (cả 2 pin torch version riêng).  
Chạy vLLM trong session sạch, lưu responses, đóng session.

## 1. Install

In [ ]:
%%capture
!pip install vllm bitsandbytes openai

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/tts_experiment"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive dir: {DRIVE_DIR}")

## 3. Upload queries file

Upload `tts_eval_queries.txt` lên Drive trước, hoặc upload trực tiếp vào Colab.

In [ ]:
# Option A: đọc từ Drive
QUERIES_FILE = f"{DRIVE_DIR}/tts_eval_queries.txt"  # ← upload file vào đây

# Option B: upload trực tiếp
# from google.colab import files
# uploaded = files.upload()  # chọn tts_eval_queries.txt
# QUERIES_FILE = "tts_eval_queries.txt"

with open(QUERIES_FILE, encoding="utf-8") as f:
    TEST_QUERIES = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(TEST_QUERIES)} queries")
print("First 3:", TEST_QUERIES[:3])

## 4. Start vLLM server

In [ ]:
import subprocess, time, requests

vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen3-4B-Instruct-2507",
        "--dtype", "half",
        "--quantization", "bitsandbytes",
        "--load-format", "bitsandbytes",
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.4",  # session sạch, không chia sẻ VRAM
        "--max-num-seqs", "4",
        "--port", "8000",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("Waiting for vLLM to start...")
for _ in range(60):
    time.sleep(5)
    try:
        if requests.get("http://localhost:8000/health", timeout=3).status_code == 200:
            print("vLLM ready!")
            break
    except Exception:
        pass
else:
    raise RuntimeError("vLLM did not start in time")

## 5. Generate responses (temperature=0)

In [ ]:
import json
from pathlib import Path
from openai import AsyncOpenAI

RESPONSES_FILE = f"{DRIVE_DIR}/llm_responses.json"

SYSTEM_PROMPT = """Bạn là chuyên gia tư vấn dinh dưỡng qua giọng nói. Tuân thủ:
1. Trả lời dựa trên kiến thức dinh dưỡng. Không trích dẫn tên nguồn hay URL.
2. Bắt đầu bằng "Chào bạn,", tư vấn như chuyên gia dinh dưỡng.
3. Ngắn gọn, dễ nghe: tối đa 100 từ, dùng câu ngắn, không dùng bullet points.
4. Kết thúc bằng "Để được tư vấn chính xác, bạn nên gặp bác sĩ dinh dưỡng."
"""

llm_client = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

async def generate_one(query: str) -> str:
    resp = await llm_client.chat.completions.create(
        model="Qwen/Qwen3-4B-Instruct-2507",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": query},
        ],
        stream=False,
        max_tokens=300,
        temperature=0,
    )
    return resp.choices[0].message.content

# Skip nếu đã có
if Path(RESPONSES_FILE).exists():
    with open(RESPONSES_FILE, encoding="utf-8") as f:
        llm_responses = json.load(f)
    print(f"Already exists: {len(llm_responses)} responses loaded from Drive")
else:
    print(f"Generating {len(TEST_QUERIES)} responses...")
    llm_responses = {}
    for i, query in enumerate(TEST_QUERIES):
        llm_responses[query] = await generate_one(query)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(TEST_QUERIES)} done")
    with open(RESPONSES_FILE, "w", encoding="utf-8") as f:
        json.dump(llm_responses, f, ensure_ascii=False, indent=2)
    print(f"Saved to Drive: {RESPONSES_FILE}")

lengths = [len(v) for v in llm_responses.values()]
print(f"Response length — min={min(lengths)} max={max(lengths)} avg={sum(lengths)//len(lengths)} chars")

## 6. Shutdown vLLM

In [ ]:
vllm_proc.terminate()
vllm_proc.wait()
print("vLLM stopped.")
print(f"\nDone. llm_responses.json saved to Drive.")
print("Tiếp theo: chạy tts_exp1_configA.ipynb / configB.ipynb / configC.ipynb")